# 🏥 Medical LLM Assistant - Notebook Organizado

**Estrutura em 6 seções independentes:**

1. **Setup** - Git, dependências, GPU (sempre execute primeiro)
2. **Configuração** - Variáveis, download do LoRA
3. **Modelo + API** - Sobe só o modelo como API (para uso remoto)
4. **Banco de Dados** - Seed e limpeza de pacientes
5. **UI Gradio** - Interface web no Colab (opcional)
6. **Publicação** - Gist de resultados

**Fluxos recomendados:**
- **API Mode (recomendado)**: 1 → 2 → 3 → 4
- **UI no Colab**: 1 → 2 → 4 → 5
- **Reset dados**: 4 (com LIMPAR=True)


## Seção 1: Setup e Dependências

Sempre execute esta célula primeiro.

In [ ]:
# 1.1 Git - Atualiza código!cd /content/medical-llm-assistant && git pull# 1.2 Dependências!pip install -q unsloth peft bitsandbytes accelerate gdown!pip install -q chromadb langgraph gradio fastapi uvicorn pyngrok# 1.3 Verificação GPUimport torchprint(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A")# 1.4 Path do projetoimport syssys.path.insert(0, '/content/medical-llm-assistant')print("✅ Setup completo")

## Seção 2: Configuração

Defina USE_MOCK e baixe o LoRA se necessário.

In [ ]:
import os# Modo de operaçãoUSE_MOCK = 'false'  # 'true' = MockLLM, 'false' = Mistral real# PathsCHROMA_DB_PATH = '/content/chroma_db'LORA_PATH = '/content/model'# Drive do Lucas (pasta pública com o LoRA)LUCAS_FOLDER_ID = '1i7SbQDLxuGIPGheTHuAgYXUQ_ZU1vIN6'# Configura ambienteos.environ['USE_MOCK_LLM'] = USE_MOCKos.environ['CHROMA_DB_PATH'] = CHROMA_DB_PATHos.environ['MODEL_PATH'] = LORA_PATHprint(f"Modo: {'Mock' if USE_MOCK == 'true' else 'Mistral Real'}")print(f"ChromaDB: {CHROMA_DB_PATH}")print(f"LoRA: {LORA_PATH}")# Download do LoRA (se modo real)if USE_MOCK != 'true':    import subprocess, glob, zipfile, os as os2    os2.makedirs(LORA_PATH, exist_ok=True)        result = subprocess.run(        ['gdown', '--folder', f'https://drive.google.com/drive/folders/{LUCAS_FOLDER_ID}',         '-O', LORA_PATH, '--remaining-ok'],        capture_output=True, text=True    )        # Extrai ZIP se houver    zips = glob.glob(f"{LORA_PATH}/*.zip")    if zips:        with zipfile.ZipFile(zips[0], 'r') as zf:            zf.extractall(LORA_PATH)        print(f"📦 Extraído: {zips[0]}")        # Valida    import os    has_config = any('adapter_config.json' in f for r, d, f in os.walk(LORA_PATH))    has_model = any('adapter_model.safetensors' in f for r, d, f in os.walk(LORA_PATH))        if has_config and has_model:        print("✅ LoRA baixado e validado")    else:        print("⚠️ Arquivos do LoRA não encontrados")else:    print("✅ Modo Mock - LoRA não necessário")

## Seção 3: Modelo + API (Modo Remoto)

Sobe apenas o modelo como API REST.

**Use quando:** quer que o huszardoBot use a UI local dele (Oracle)

Ao executar, copie a URL do ngrok e cole no chat.

In [ ]:
# Esta célula sobe APENAS o modelo como API# Você pode rodar isso e depois usar a UI local no Oracleimport osimport syssys.path.insert(0, '/content/medical-llm-assistant')from fastapi import FastAPIfrom pydantic import BaseModelimport uvicornfrom threading import Threadimport nest_asyncionest_asyncio.apply()# Carrega modeloprint("🔄 Carregando modelo...")from src.llm.factory import get_llmfrom src.rag.patient_rag import get_patientfrom src.graph.pipeline import run_consultation_llm = get_llm()print(f"✅ Modelo carregado: {type(_llm).__name__}")# APIapp_api = FastAPI(title="Medical LLM API")class ConsultRequest(BaseModel):    cpf: str    question: str@app_api.post("/consult")def consult(req: ConsultRequest):    """Endpoint principal de consulta médica."""    profile = get_patient(req.cpf)    if not profile:        return {            "error": "Paciente não encontrado",            "cpf": req.cpf        }        result = run_consultation(        cpf=req.cpf,        doctor_question=req.question,        llm=_llm,        patient_profile=profile    )        return {        "cpf": req.cpf,        "profile": result.get("patient_profile"),        "answer": result.get("final_answer"),        "safety_passed": result.get("safety_passed"),        "needs_escalation": result.get("needs_escalation")    }@app_api.get("/health")def health():    return {"status": "ok", "model": type(_llm).__name__}# Expõe via ngrokfrom pyngrok import ngrokpublic_url = ngrok.connect(8000)print(f"🌐 API URL: {public_url}")print(f"   Health: {public_url}/health")print(f"   Consult: {public_url}/consult")print("\n💡 Copie a URL acima e cole no chat para o huszardoBot usar")# Roda servidordef run():    uvicorn.run(app_api, host="0.0.0.0", port=8000)Thread(target=run, daemon=True).start()print("\n✅ API rodando em background")

## Seção 4: Banco de Dados

Seed de pacientes e limpeza.

In [ ]:
import osimport shutilos.environ['CHROMA_DB_PATH'] = '/content/chroma_db'import syssys.path.insert(0, '/content/medical-llm-assistant')from src.rag.patient_rag import seed_from_file# Opção A: Limpar e recriarLIMPAR = False  # Mude para True para limparif LIMPAR and os.path.exists('/content/chroma_db'):    shutil.rmtree('/content/chroma_db')    print("🗑️ ChromaDB limpo")# Seed de pacientesn = seed_from_file('/content/medical-llm-assistant/data/patients_seed.json')print(f"✅ {n} pacientes carregados")# Lista pacientesfrom src.rag.patient_rag import _get_clientclient = _get_client()collection = client.get_collection("patients")print(f"📊 Total no DB: {collection.count()}")

## Seção 5: UI Gradio (Opcional)

Interface web completa no Colab.

**Use quando:** quer usar diretamente no navegador (sem depender do bot).

In [ ]:
# Só execute se quiser usar a interface web diretamente no Colab# Se estiver usando a API (Seção 3), pode pular estaimport osimport syssys.path.insert(0, '/content/medical-llm-assistant')os.environ['CHROMA_DB_PATH'] = '/content/chroma_db'from app import demodemo.launch(    share=True,  # Gera link público    debug=True,    show_error=True)

## Seção 6: Publicação

Publica audit log no Gist.

In [ ]:
# Publica resultados de testes no Gistimport jsonimport requestsimport osfrom datetime import datetimeGIST_ID = 'f3e6e10d65eff30560abf756467d8d1b'GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')  # Defina no secrets se necessário# Lê audit logaudit_path = '/tmp/medical_audit.jsonl'if os.path.exists(audit_path):    with open(audit_path) as f:        lines = [json.loads(l) for l in f if l.strip()]        # Prepara conteúdo    content = '\\n'.join(json.dumps(l, ensure_ascii=False) for l in lines[-50:])  # últimos 50        # Atualiza Gist    url = f"https://api.github.com/gists/{GIST_ID}"    headers = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}    data = {        "files": {            "audit_log.jsonl": {                "content": content            }        }    }        resp = requests.patch(url, headers=headers, json=data)    if resp.status_code == 200:        print(f"✅ Audit log atualizado: https://gist.github.com/felipe-huszar/{GIST_ID}")    else:        print(f"⚠️ Erro: {resp.status_code}")        print(resp.text)else:    print("⚠️ Audit log não encontrado")